# Second Experiment: Standalone-Submission Grading

Same real corpus (4 exercises x 3 rounds x 100 instances = 1,200 trajectories,
6,000 images), graded a **second, independent** time under
`partial_credit_policy="standalone_submission"` (see `grading/prompts.py` and
the README's "Second experiment" section) instead of the original
`"step_calibrated"` policy. The first experiment's code, data, and notebook
(`notebooks/statistical_analysis.ipynb`) are untouched by this.

Under `standalone_submission`, each image is graded as if it were a complete,
standalone answer: no reference image shown, no step number or expected-
progress hint, and partial credit justified strictly by closeness to each
rubric rule's own quantity/relation/expected threshold (no leniency for
"it's early, it's moving the right direction"). The question this answers is
not "is the grader too lenient about correction progress" (experiment 1) but
"judged purely on its own merits, does each step actually look like an
increasingly-correct standalone answer -- and if not, what does that reveal
about the generation method?"

Run from the `diffusion_module_v2/` project root.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kinematics_grading.analysis.acceptance import (
    acceptance_summary_by_exercise,
    first_accepted_step_distribution,
    monotonicity_summary_by_exercise,
    partial_credit_on_unsatisfied_rules,
    per_step_pass_rate,
    trajectory_score_monotonicity,
    trajectory_summary,
)
from kinematics_grading.analysis.embeddings import EmbeddingCache
from kinematics_grading.config import DATA_DIR, load_settings
from kinematics_grading.domain import parse_step_info

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
GRADING_STANDALONE = ROOT / "output" / "grading_results_standalone"
GRADING_STEP_CALIBRATED = ROOT / "output" / "grading_results"
ANALYSIS_DIR = ROOT / "output" / "analysis"
FIG_DIR = ANALYSIS_DIR / "notebook_figures_standalone"
FIG_DIR.mkdir(parents=True, exist_ok=True)

settings = load_settings()
SCORE_THRESHOLD = settings.analysis.score_threshold
print("SCORE_THRESHOLD:", SCORE_THRESHOLD)

In [ ]:
def load_graded(jsonl_path):
    d = pd.read_json(jsonl_path, lines=True)
    d["step_order"] = d["step_file"].apply(lambda s: parse_step_info(s).order if parse_step_info(s) else None)
    d = d.dropna(subset=["step_order"])
    d["step_order"] = d["step_order"].astype(int)
    return d

df = load_graded(GRADING_STANDALONE / "intermediate_step_rule_grades.jsonl")
print(f"{len(df)} graded images (standalone_submission policy)")

# Same generated images as experiment 1 -> same embeddings, reuse the existing
# cache instead of recomputing (dist_to_ref depends only on pixel content).
cache = EmbeddingCache(ANALYSIS_DIR / "embeddings_cache.npz")
dists = []
for ex, group in df.groupby("exercise"):
    ref_path = DATA_DIR / f"exercice_{int(ex)}" / f"correct_{int(ex)}.png"
    ref_emb = cache.get(ref_path)
    for _, row in group.iterrows():
        dists.append(EmbeddingCache.cosine_distance(cache.get(Path(row["generated_image"])), ref_emb))
df["dist_to_ref"] = dists
cache.save()  # any images not already cached (there shouldn't be any -- same corpus) get persisted

traj_df = trajectory_summary(df, SCORE_THRESHOLD)
print(f"{len(traj_df)} trajectories graded so far")
traj_df.head()

## 1. Acceptance summary (standalone_submission)

In [ ]:
acc_summary = acceptance_summary_by_exercise(traj_df)
overall_pct = 100.0 * traj_df["has_acceptance"].mean()
print(f"Whole corpus so far: {traj_df['has_acceptance'].sum()}/{len(traj_df)} accepted ({overall_pct:.2f}%)")
acc_summary.round(2)

## 2. Per-step pass rate (standalone_submission)

In [ ]:
step_rates = per_step_pass_rate(df, SCORE_THRESHOLD)

fig, ax = plt.subplots(figsize=(8, 5))
for ex, g in step_rates.groupby("exercise"):
    g = g.sort_values("step_order")
    ax.plot(g["step_order"], g["pct_pass"], marker="o", label=f"Exercise {int(ex)}")
ax.axhline(50, linestyle="--", linewidth=1, color="gray", label="50% of trajectories")
ax.set_xlabel("Correction step"); ax.set_ylabel("% of images passing (score >= 0.60)")
ax.set_title("Per-step pass rate by exercise (standalone_submission)")
ax.set_xticks([1, 2, 3, 4, 5]); ax.set_ylim(0, 105)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "per_step_pass_rate_standalone.png", dpi=150, bbox_inches="tight")
plt.show()

step_rates.pivot(index="step_order", columns="exercise", values="pct_pass").round(1)

## 3. First-accepted-step distribution (standalone_submission)

In [ ]:
first_dist = first_accepted_step_distribution(traj_df)
pivot = first_dist.assign(step=first_dist["step"].apply(lambda s: "never" if pd.isna(s) else int(s)))
pivot = pivot.pivot(index="exercise", columns="step", values="pct_of_trajectories")
pivot = pivot[[c for c in [1, 2, 3, 4, 5, "never"] if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind="bar", ax=ax, width=0.8)
ax.set_xlabel("Exercise"); ax.set_ylabel("% of trajectories first accepted here")
ax.set_title("First-accepted-step distribution by exercise (standalone_submission)")
ax.legend(title="Step", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "first_accepted_step_distribution_standalone.png", dpi=150, bbox_inches="tight")
plt.show()

pivot.round(1)

## 4. Partial credit from unsatisfied rules (standalone_submission)

Same metric as experiment 1's core finding, recomputed here. If the prompt redesign worked, these numbers should be far lower than experiment 1's (27-91% depending on exercise) -- a wrong-sign rule should now get ~0 credit instead of generous partial credit.

In [ ]:
partial_all = partial_credit_on_unsatisfied_rules(df, SCORE_THRESHOLD)
partial_early = partial_credit_on_unsatisfied_rules(df[df["step_order"] <= 2], SCORE_THRESHOLD)
# outer join: an exercise with zero early (steps 1-2) passing records -- e.g.
# Exercise 4 here, which passes 0% of the time before step 5 -- is itself a
# real finding, not something to silently drop from the comparison.
comparison = partial_all.merge(partial_early, on="exercise", suffixes=("_all_steps", "_steps_1_2"), how="outer")
comparison.round(2)

## 5. Score variance / monotonicity: judging the generation method

The point of this experiment. Under `step_calibrated`'s leniency, scores climb almost monotonically by construction. Graded as independent standalone submissions, does each step still look increasingly correct on its own merits -- or does the generation method produce steps that are sometimes a *worse* standalone answer than the step before?

In [ ]:
mono_df = trajectory_score_monotonicity(df)
mono_summary = monotonicity_summary_by_exercise(mono_df)
print(f"Corpus-wide: {100*mono_df['is_monotonic_nondecreasing'].mean():.1f}% of trajectories are monotonic "
      f"non-decreasing so far; {(~mono_df['is_monotonic_nondecreasing']).sum()} have at least one regression.")
mono_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(mono_summary["exercise"].astype(str), mono_summary["pct_with_any_regression"])
ax.set_xlabel("Exercise"); ax.set_ylabel("% of trajectories with >=1 score regression")
ax.set_title("Step-to-step score regressions under standalone grading")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "regression_rate_by_exercise.png", dpi=150, bbox_inches="tight")
plt.show()

# A concrete example regression, if any exist yet.
regressed = mono_df[~mono_df["is_monotonic_nondecreasing"]].sort_values("max_drop", ascending=False)
if not regressed.empty:
    r = regressed.iloc[0]
    example = df[
        (df["exercise"] == r["exercise"]) & (df["round"] == r["round"])
        & (df["source_exercise"] == r["source_exercise"]) & (df["instance"] == r["instance"])
    ].sort_values("step_order")
    print(f"Largest regression so far: Ex{int(r['exercise'])} {r['instance']}, drop={r['max_drop']:.2f}")
    print(example[["step_file", "score_ratio", "feedback"]].to_string(index=False))
else:
    print("No regressions found yet (partial data -- re-run once grading completes).")

## 6. Comparison against experiment 1 (`step_calibrated`)

Loads the first experiment's real graded corpus (`output/grading_results/intermediate_step_rule_grades.jsonl`) and recomputes the same metrics for a direct, apples-to-apples, same-corpus comparison.

In [ ]:
# dist_to_ref isn't needed for this comparison (acceptance rate, per-step pass
# rate, and monotonicity are all score-only); trajectory_summary() handles its
# absence gracefully (first_accepted_dist_to_ref just comes back None).
df_step_calibrated = load_graded(GRADING_STEP_CALIBRATED / "intermediate_step_rule_grades.jsonl")
traj_step_calibrated = trajectory_summary(df_step_calibrated, SCORE_THRESHOLD)

step_rates_calibrated = per_step_pass_rate(df_step_calibrated, SCORE_THRESHOLD)
mono_calibrated = trajectory_score_monotonicity(df_step_calibrated)
mono_summary_calibrated = monotonicity_summary_by_exercise(mono_calibrated)

print("=== Overall acceptance ===")
print(f"step_calibrated:        {traj_step_calibrated['has_acceptance'].mean()*100:.2f}%")
print(f"standalone_submission:  {traj_df['has_acceptance'].mean()*100:.2f}% (of {len(traj_df)} graded so far)")

print()
print("=== % of trajectories with >=1 score regression (step-to-step) ===")
compare_mono = mono_summary_calibrated[["exercise", "pct_with_any_regression"]].merge(
    mono_summary[["exercise", "pct_with_any_regression"]], on="exercise",
    suffixes=("_step_calibrated", "_standalone_submission"),
)
compare_mono.round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
for ax, ex in zip(axes, [1, 2, 3, 4]):
    g1 = step_rates_calibrated[step_rates_calibrated["exercise"] == ex].sort_values("step_order")
    g2 = step_rates[step_rates["exercise"] == ex].sort_values("step_order")
    ax.plot(g1["step_order"], g1["pct_pass"], marker="o", label="step_calibrated")
    ax.plot(g2["step_order"], g2["pct_pass"], marker="s", label="standalone_submission")
    ax.axhline(50, linestyle="--", linewidth=0.8, color="gray")
    ax.set_title(f"Exercise {ex}"); ax.set_xlabel("Step"); ax.set_xticks([1,2,3,4,5])
    ax.grid(alpha=0.3)
axes[0].set_ylabel("% passing"); axes[0].legend(fontsize=8)
fig.suptitle("Per-step pass rate: step_calibrated vs. standalone_submission", y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / "policy_comparison_per_step_pass_rate.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Whether the partial-credit-on-unsatisfied-rules mechanism (experiment 1's
  headline finding) is a prompt artifact: compare Section 4's numbers against
  the first notebook's (Ex.1 27%, Ex.2 30%, Ex.3 55%, Ex.4 91%).
- Whether the generation method produces monotonically-improving standalone
  answers, or whether some steps are genuine standalone regressions: Section
  5/6's regression rates, above zero for `step_calibrated`'s (leniency-
  driven) near-monotonic climb.
- This notebook re-derives everything from `output/grading_results_standalone/`
  and reuses the tested `analysis/acceptance.py` functions (including the two,
  `trajectory_score_monotonicity` / `monotonicity_summary_by_exercise`, added
  specifically for this second experiment) -- nothing here is retyped by hand.
